In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_json("../raw/diary-answer-20251010_20251126.json")

In [3]:
# 2. 설문 데이터(isi, ssi, phq-9)에서 최신 Score 추출 함수
def get_latest_score(survey_list):
    if isinstance(survey_list, list) and len(survey_list) > 0:
        # 가장 마지막(최신) 데이터의 score 반환
        return survey_list[-1].get('score')
    return None

# 각 설문 점수 컬럼 생성
df_raw['isi_score'] = df_raw['isi'].apply(get_latest_score)
df_raw['ssi_score'] = df_raw['ssi'].apply(get_latest_score)
df_raw['phq9_score'] = df_raw['phq-9'].apply(get_latest_score)

# 3. diaries 컬럼을 행으로 펼치기 (Explode)
# 일기 데이터가 없는 유저(None)를 처리하고 리스트를 개별 행으로 분리합니다.
df_exploded = df_raw.explode('diaries').dropna(subset=['diaries'])

# 4. 펼쳐진 diaries 객체에서 상세 정보 추출
# 리스트에서 풀린 딕셔너리 데이터를 개별 컬럼으로 변환
diaries_df = pd.json_normalize(df_exploded['diaries'])
diaries_df.index = df_exploded.index # 인덱스 동기화

# 5. 유저 정보와 일기 상세 정보 결합
df_final = pd.concat([
    df_exploded[['user_id', 'gender', 'isi_score', 'ssi_score', 'phq9_score']], 
    diaries_df[['status', 'content', 'log_date']]
], axis=1)

# 6. status가 'ANSWERED'인 데이터만 필터링
result_table = df_final[df_final['status'] == 'ANSWERED'].copy()

# 7. 원하는 순서대로 컬럼 정렬 및 결과 확인
result_table = result_table[['user_id', 'gender', 'log_date', 'content', 'isi_score', 'ssi_score', 'phq9_score']]

In [4]:
result_table = result_table.dropna(subset=['phq9_score'])

In [5]:
result_table.to_csv("./preprocess_1/preprocess_1.csv")